# Imports

In [ ]:
import importlib.metadata
import warnings
from datetime import datetime

import pandas as pd
from diive.core.io.filereader import search_files, MultiDataFileReader
from diive.core.io.files import save_parquet

# pd.options.display.width = None
# pd.options.display.max_columns = None
pd.set_option('display.max_rows', 3000)
pd.set_option('display.max_columns', 3000)
warnings.filterwarnings(action='ignore', category=FutureWarning)
warnings.filterwarnings(action='ignore', category=UserWarning)
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
version_diive = importlib.metadata.version("diive")
print(f"diive version: v{version_diive}")

In [ ]:
# Project settings (dataset period, data paths): see src/config.py
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import DATA_DIR, PERIOD_START, PERIOD_END, select_period
print(f"Dataset period: {PERIOD_START} to {PERIOD_END} (CET, UTC+1)")

# Merge L1 results to parquet

In [ ]:
# Search original files and store filepaths in list
SOURCEDIR = str(DATA_DIR / "FLUXES" / "LGR")
sourcefiles = search_files(searchdirs=SOURCEDIR, pattern='*')
sourcefiles

In [ ]:
# Read data from files to dataframe
d = MultiDataFileReader(filepaths=sourcefiles,
                        filetype='EDDYPRO-FLUXNET-CSV-30MIN',
                        output_middle_timestamp=True)
df = d.data_df
df

# Restrict to dataset period

17 Oct 2023 to 22 Aug 2024, see `src/config.py`. EC measurements start on 7 Nov 2023.

In [ ]:
df = select_period(df, timestamp='middle')
df

# Remove vars with zero records

In [ ]:
stats = df.describe().T

In [ ]:
# Find available variables that have more than 0 records, remove all others from data
available_vars = stats.loc[stats['count'] > 0].index
not_available_vars = stats.loc[stats['count'] == 0].index
print("No records for the following vars:")
print(stats.loc[not_available_vars])

In [ ]:
print("Removing variables with 0 records ...")
df_available = df[available_vars].copy()

print(f"Variables removed:")
df_not_available = df[not_available_vars].copy()
_list = df_not_available.columns.tolist()
for c in _list:
    print(c, end=" ")

# Remove meteo variables from EC L1 output

In [ ]:
removecols = [v for v in df_available if str(v).endswith("_1_1_1")]
print("The following variables will be removed from EC results:")
print(removecols)

In [ ]:
# Remove meteo data from flux results because this meteo has gaps after flux processing
keepcols = []
[keepcols.append(c) for c in df_available.columns if "_1_1_1" not in c]
df_reduced = df_available[keepcols].copy()
df_reduced

# Save to file

In [ ]:
OUTNAME = "32.1_CH-TAN_LGR_Level-1_eddypro_fluxnet"
filepath = save_parquet(filename=OUTNAME, data=df_reduced)

# **End of notebook**.

In [ ]:
dt_string = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"Finished. {dt_string}")